In [ ]:
# Print current md-python tag
import importlib.metadata
import json
import subprocess

direct_url_files = [p for p in importlib.metadata.files('md-python') if 'direct_url.json' in str(p)]
if direct_url_files:
    direct_url = json.loads(direct_url_files[0].read_text())
    url = direct_url.get('url', '')
    if 'tags/' in url:
        tag = url.rstrip('.tar.gz').split('/')[-1]
    elif url.startswith('file://'):
        repo_path = url[len('file://'):]    
        result = subprocess.run(['git', 'describe', '--tags'], cwd=repo_path, capture_output=True, text=True)
        tag = result.stdout.strip() or '(no tag)'
    else:
        tag = '(unknown)'
    print(tag)
else:
    print('(no install metadata found)')

In [ ]:
# Setup: load env, create client
import os
from dotenv import load_dotenv
from md_python import MDClient

def create_md_client_from_env() -> MDClient:
    load_dotenv()
    api_token = os.getenv('MD_AUTH_TOKEN')
    assert api_token, 'MD_AUTH_TOKEN not set (check your .env)'
    base_url = os.getenv('MD_API_BASE_URL')
    return MDClient(api_token=api_token, base_url=base_url)

client = create_md_client_from_env()
print('Client initialized.')

In [ ]:
# Set the upload ID from a previous upload
upload_id = "76f2b5ad-2cc1-4a1b-aaf5-c71a40534343"  # <-- set this to your upload ID

In [ ]:
# Load experiment design and sample metadata from CSV
import csv
from pathlib import Path
from typing import List, Tuple
from md_python import ExperimentDesign, SampleMetadata

LOCAL_DATA_DIR = Path('/Users/giuseppeinfusini/wd/md-repos/md-python/development/GI/do_not_save/test_data').resolve()
DESIGN_CSV_NAME = 'experiment_design_COMBINED.csv'

def _header_lower(header):
    return [h.strip().lower() if isinstance(h, str) else '' for h in header]

def _find_col(hl, names):
    for n in names:
        if n in hl:
            return hl.index(n)
    raise ValueError(f'CSV must have one of {names}')

def load_design_and_metadata(csv_path: Path) -> Tuple[ExperimentDesign, SampleMetadata]:
    with open(csv_path, 'r', encoding='utf-8') as f:
        raw = list(csv.reader(f))
    header = [h.strip() for h in raw[0]]
    hl = _header_lower(header)
    idx_fn     = _find_col(hl, ('filename', 'file'))
    idx_sample = _find_col(hl, ('sample_name', 'sample'))
    idx_cond   = _find_col(hl, ('condition', 'group'))
    data_rows  = [r for r in raw[1:] if r]

    design_data = [['filename', 'sample_name', 'condition']]
    for r in data_rows:
        design_data.append([
            r[idx_fn].strip()     if len(r) > idx_fn     else '',
            r[idx_sample].strip() if len(r) > idx_sample else '',
            r[idx_cond].strip()   if len(r) > idx_cond   else '',
        ])

    sample_col_indices = [i for i, h in enumerate(hl) if h not in ('filename', 'file')]
    sample_headers = [header[i] for i in sample_col_indices]
    seen, sample_rows = set(), []
    for r in data_rows:
        sn = r[idx_sample].strip() if len(r) > idx_sample else ''
        if sn and sn not in seen:
            seen.add(sn)
            sample_rows.append([r[i].strip() if len(r) > i else '' for i in sample_col_indices])

    return ExperimentDesign(data=design_data), SampleMetadata(data=[sample_headers] + sample_rows)

design_csv_path = LOCAL_DATA_DIR / DESIGN_CSV_NAME
assert design_csv_path.exists(), f'Missing {DESIGN_CSV_NAME} in {LOCAL_DATA_DIR}'
exp_design, sample_md = load_design_and_metadata(design_csv_path)
print(f'Design rows: {len(exp_design.data) - 1}, Sample rows: {len(sample_md.data) - 1}')

In [ ]:
# Find the initial intensity dataset for this upload
assert upload_id, 'Set upload_id in the cell above'
dataset = client.datasets.find_initial_dataset(upload_id)
assert dataset is not None, f'No initial dataset found for upload {upload_id}'
print('Initial dataset ID:', dataset.id)

In [ ]:
# Inspect sample metadata columns — set CONDITION_COLUMN below
print("Sample metadata columns:", sample_md.data[0])
print()
print("First few rows:")
for row in sample_md.data[1:4]:
    print(" ", row)

In [ ]:
# Set the column to use for pairwise comparisons
CONDITION_COLUMN = "condition"  # <-- set to one of the columns shown above

# Generate all pairwise comparisons from the column
from md_python import PairwiseComparisonDataset

comparisons = PairwiseComparisonDataset.all_pairwise_comparisons(sample_md, CONDITION_COLUMN)
print(f"All pairwise comparisons for column '{CONDITION_COLUMN}' ({len(comparisons)} total):")
for c in comparisons:
    print(f"  {c[0]} vs {c[1]}")

In [ ]:
# Submit a single PairwiseComparisonDataset with all comparisons
from md_python import PairwiseComparisonDataset

dataset_name = f"Pairwise_{CONDITION_COLUMN}"

builder = PairwiseComparisonDataset(
    input_dataset_ids=[str(dataset.id)],
    dataset_name=dataset_name,
    sample_metadata=sample_md,
    condition_column=CONDITION_COLUMN,
    condition_comparisons=comparisons,
    entity_type="protein",
)

pw_id = builder.run(client)
print(f"Submitted {dataset_name} → dataset_id={pw_id}")

In [ ]:
# Poll until the pairwise job reaches a terminal state
result = client.datasets.wait_until_complete(
    upload_id=upload_id,
    dataset_id=str(pw_id),
    poll_s=10,
    timeout_s=3600,
)
print(f"state={result.state}")
result